# EMG — LMS Adaptive Powerline Canceller

**Pipeline** (all channels in the selected folder):
1. Raw signal
2. → Bandpass filter (6–500 Hz, 4th-order Butterworth, zero-phase SOS)
3. → **Block NLMS adaptive filter** — estimates and subtracts the powerline interference

**How the LMS works:**  
A synthetic reference is built from `sin + cos` at `f0, 2f0, … n_harmonics × f0`.  
The filter finds weights `w` such that `ref @ w ≈ interference`, then returns `e = signal − ref @ w`.  
Using both sin and cos per harmonic lets the filter match any amplitude and phase.  
Weights are updated once per block (Block NLMS) — fully vectorised, fast.

**±1 Hz accuracy:** set `f0` to the actual mains frequency with the slider (49–51 Hz). The NLMS weights adapt continuously, so small slow frequency drift is tracked automatically.

**Output — 2 Plotly figures per channel:**
- **Fig A** — 3 rows × 2 cols: Raw / Bandpass / LMS × time domain | FFT
- **Fig B** — FFT overlay of all three stages with harmonic markers

Run cells top to bottom, then use the widget in the last cell.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import signal as sig

import plotly.graph_objects as go
from plotly.subplots import make_subplots

import ipywidgets as widgets
from IPython.display import display, clear_output

RECORDINGS_DIR = Path("recordings")

In [ ]:
# ── data ─────────────────────────────────────────────────────────────────────

def load_signal(folder: str, channel: str):
    path = RECORDINGS_DIR / folder / f"{channel}.csv"
    df = pd.read_csv(path)
    t = df["time"].values
    x = df["data"].values
    fs = round(1.0 / (t[1] - t[0]))
    return t, x, float(fs)

# ── filters ───────────────────────────────────────────────────────────────────

def bandpass_filter(x: np.ndarray, fs: float,
                    low_hz: float = 6.0, high_hz: float = 500.0,
                    order: int = 4) -> np.ndarray:
    """Zero-phase Butterworth bandpass — SOS form for numerical stability."""
    sos = sig.butter(order, [low_hz, high_hz], btype="band", fs=fs, output="sos")
    return sig.sosfiltfilt(sos, x)


def lms_filter(x: np.ndarray, fs: float,
               f0: float = 50.0,
               mu: float = 0.1,
               n_harmonics: int = 5,
               block_size: int = 256) -> np.ndarray:
    """Block Normalised LMS (NLMS) adaptive powerline interference canceller.

    Parameters
    ----------
    x           : input signal (bandpass-filtered EMG + interference)
    fs          : sample rate (Hz)
    f0          : reference fundamental — set to the actual mains frequency (Hz)
    mu          : NLMS step size (0 < mu < 2; larger = faster convergence but noisier)
    n_harmonics : number of harmonics cancelled  (f0, 2f0 … n_harmonics*f0)
    block_size  : samples per weight-update step (larger = faster, less adaptive)

    Returns
    -------
    e : cleaned signal  (error = input − estimated interference)

    Algorithm
    ---------
    Reference matrix R  shape (N, 2*n_harmonics):
      columns = sin(2π k f0 t), cos(2π k f0 t)  for k = 1 … n_harmonics

    Per block B:
      y   = R[B] @ w          # estimated interference
      e   = x[B] - y          # cleaned EMG
      w  += (mu / ||R[B]||²) * R[B].T @ e   # NLMS weight update
    """
    N   = len(x)
    n_w = 2 * n_harmonics

    # Build reference: alternating sin/cos columns for each harmonic
    t   = np.arange(N) / fs
    ref = np.column_stack([
        fn(2 * np.pi * k * f0 * t)
        for k in range(1, n_harmonics + 1)
        for fn in (np.sin, np.cos)
    ])                                         # (N, n_w)

    w   = np.zeros(n_w)                        # adaptive weights
    e   = np.zeros(N)                          # output buffer
    eps = 1e-8                                 # regularisation

    for start in range(0, N, block_size):
        end  = min(start + block_size, N)
        R    = ref[start:end]                  # (B, n_w)
        y    = R @ w                           # (B,)  estimated interference
        e_b  = x[start:end] - y               # (B,)  error = cleaned EMG
        e[start:end] = e_b
        norm = np.sum(R ** 2) + eps            # Frobenius norm²
        w   += (mu / norm) * (R.T @ e_b)      # NLMS weight update

    return e

# ── FFT ───────────────────────────────────────────────────────────────────────

def rfft_single_sided(x: np.ndarray, fs: float):
    N     = len(x)
    freqs = np.fft.rfftfreq(N, d=1.0 / fs)
    mag   = np.abs(np.fft.rfft(x)) * (2.0 / N)
    mag[0] /= 2.0
    if N % 2 == 0:
        mag[-1] /= 2.0
    return freqs, mag

# ── palette ───────────────────────────────────────────────────────────────────

STAGE_STYLES = [
    # (label, line_color, fill_color, time_key, fft_key)
    ("Raw",               "#1565C0", "rgba(21,101,192,0.10)",  "raw", "fft_raw"),
    ("Bandpass 6\u2013500 Hz", "#E65100", "rgba(230,81,0,0.10)",    "bp",  "fft_bp"),
    ("Bandpass + LMS",    "#2E7D32", "rgba(46,125,50,0.10)",   "lms", "fft_lms"),
]

CHANNELS = [
    ("emg_line_L", "Line In \u2014 Left"),
    ("emg_line_R", "Line In \u2014 Right"),
    ("emg_mic",    "Microphone"),
]

# ── main ──────────────────────────────────────────────────────────────────────

def analyze_lms(folder: str, fft_max_hz: float = 1000.0,
                f0: float = 50.0, mu: float = 0.1,
                n_harmonics: int = 5, block_size: int = 256):
    """Load every channel in *folder*, apply bandpass + LMS, and plot."""

    lms_tag = f"f0={f0:.1f} Hz  \u03bc={mu:.4f}  harmonics={n_harmonics}"

    stage_styles = [
        STAGE_STYLES[0],
        STAGE_STYLES[1],
        (f"Bandpass + LMS  ({lms_tag})", *STAGE_STYLES[2][1:]),
    ]

    # ── load & process all channels ───────────────────────────────────────────
    chs = []
    for ch_key, ch_label in CHANNELS:
        path = RECORDINGS_DIR / folder / f"{ch_key}.csv"
        if not path.exists():
            print(f"  Skipping {ch_label}: {ch_key}.csv not found")
            continue
        print(f"Loading {ch_key}.csv \u2026", end="  ")
        t, raw, fs = load_signal(folder, ch_key)
        print(f"fs={fs:.0f} Hz  {len(raw):,} samples  {t[-1]:.3f} s")

        print(f"  Bandpass 6\u2013500 Hz \u2026")
        bp  = bandpass_filter(raw, fs)

        print(f"  LMS  ({lms_tag}  block={block_size}) \u2026")
        lms = lms_filter(bp, fs, f0=f0, mu=mu,
                         n_harmonics=n_harmonics, block_size=block_size)

        fr, fft_raw = rfft_single_sided(raw, fs)
        _,  fft_bp  = rfft_single_sided(bp,  fs)
        _,  fft_lms = rfft_single_sided(lms, fs)

        chs.append(dict(
            label=ch_label,
            time=t, raw=raw, bp=bp, lms=lms,
            fr=fr, fft_raw=fft_raw, fft_bp=fft_bp, fft_lms=fft_lms,
        ))

    if not chs:
        print("No channel files found in", folder)
        return

    n_harm_lines = min(int(fft_max_hz / f0), 20)

    for cd in chs:
        fr = cd["fr"]
        fm = fr <= fft_max_hz

        # ── Figure A: 3 rows (stages) × 2 cols (time | FFT) ──────────────────
        lms_row_title = f"LMS ({lms_tag})"
        fig_a = make_subplots(
            rows=3, cols=2,
            subplot_titles=[
                "Raw \u2014 time",               "Raw \u2014 FFT",
                "Bandpass 6\u2013500 Hz \u2014 time", "Bandpass 6\u2013500 Hz \u2014 FFT",
                f"{lms_row_title} \u2014 time",  f"{lms_row_title} \u2014 FFT",
            ],
            column_widths=[0.55, 0.45],
            vertical_spacing=0.10,
            horizontal_spacing=0.08,
        )

        for r, (sl, color, fill, sig_key, fft_key) in enumerate(stage_styles, 1):
            fig_a.add_trace(go.Scatter(
                x=cd["time"], y=cd[sig_key],
                name=sl, line=dict(width=0.6, color=color),
                showlegend=False,
            ), row=r, col=1)
            fig_a.add_trace(go.Scatter(
                x=fr[fm], y=cd[fft_key][fm],
                name=sl, line=dict(width=1.0, color=color),
                fill="tozeroy", fillcolor=fill,
                showlegend=False,
            ), row=r, col=2)
            fig_a.update_xaxes(title_text="Time (s)",       row=r, col=1)
            fig_a.update_xaxes(title_text="Frequency (Hz)", row=r, col=2)
            fig_a.update_yaxes(title_text="Amplitude (V)",  row=r, col=1)
            fig_a.update_yaxes(title_text="Magnitude (V)",  row=r, col=2)

        fig_a.update_layout(
            title_text=f"{cd['label']} \u2014 {folder}",
            height=700,
            template="plotly_white",
        )
        fig_a.show()

        # ── Figure B: FFT overlay + harmonic markers ───────────────────────────
        fig_b = go.Figure()
        for sl, color, _, _, fft_key in stage_styles:
            fig_b.add_trace(go.Scatter(
                x=fr[fm], y=cd[fft_key][fm],
                name=sl, line=dict(width=1.2, color=color),
            ))

        for k in range(1, n_harm_lines + 1):
            fig_b.add_vline(
                x=k * f0,
                line=dict(color="rgba(180,0,0,0.25)", width=1, dash="dot"),
                annotation_text=f"{k*f0:.0f}",
                annotation_position="top right",
                annotation_font_size=8,
                annotation_font_color="rgba(180,0,0,0.55)",
            )

        fig_b.update_layout(
            title_text=f"{cd['label']} \u2014 FFT Overlay \u2014 {folder}  [{lms_tag}]",
            xaxis_title="Frequency (Hz)",
            yaxis_title="Magnitude (V)",
            height=400,
            template="plotly_white",
            legend=dict(orientation="h", y=1.12, x=0.5, xanchor="center"),
        )
        fig_b.show()

    print("Done.")

In [ ]:
# ── widget UI ─────────────────────────────────────────────────────────────────
# Uses _lms_* singleton names so this notebook can coexist with emg_analysis.ipynb
# in the same kernel without variable collisions.

clear_output(wait=True)

folders = sorted([f.name for f in RECORDINGS_DIR.iterdir() if f.is_dir()])

if not folders:
    print("No recording folders found in 'recordings/'. Run dual_acquisition.py first.")
else:
    try:
        _lms_run_btn          # already exists → re-run: just refresh folder list
    except NameError:
        _lms_folder_w = widgets.Dropdown(
            options=folders,
            value=folders[-1],
            description="Recording:",
            style={"description_width": "initial"},
            layout=widgets.Layout(width="440px"),
        )
        _lms_f0_w = widgets.FloatSlider(
            value=50.0, min=49.0, max=51.0, step=0.1,
            description="Ref freq f\u2080 (Hz):",
            style={"description_width": "initial"},
            layout=widgets.Layout(width="420px"),
            readout_format=".1f",
        )
        _lms_mu_w = widgets.FloatLogSlider(
            value=0.1, base=10, min=-3, max=0, step=0.05,
            description="Step size \u03bc:",
            style={"description_width": "initial"},
            layout=widgets.Layout(width="420px"),
            readout_format=".4f",
        )
        _lms_harm_w = widgets.IntSlider(
            value=5, min=1, max=10, step=1,
            description="Harmonics:",
            style={"description_width": "initial"},
            layout=widgets.Layout(width="420px"),
        )
        _lms_block_w = widgets.Dropdown(
            options=[("1 (sample-by-sample)", 1), ("64", 64),
                     ("256 (default)", 256), ("1024", 1024),
                     ("4096", 4096)],
            value=256,
            description="Block size:",
            style={"description_width": "initial"},
            layout=widgets.Layout(width="280px"),
        )
        _lms_fft_w = widgets.BoundedIntText(
            value=1000, min=100, max=24000, step=50,
            description="FFT max Hz:",
            style={"description_width": "initial"},
            layout=widgets.Layout(width="220px"),
        )
        _lms_run_btn = widgets.Button(
            description="\u25b6  Analyze & Plot",
            button_style="primary",
            layout=widgets.Layout(width="190px", height="38px"),
        )
        _lms_out = widgets.Output()
    else:
        _lms_folder_w.options = folders
        _lms_folder_w.value   = folders[-1]

    def _lms_on_run(b):
        with _lms_out:
            clear_output(wait=True)
            analyze_lms(
                _lms_folder_w.value,
                fft_max_hz  = float(_lms_fft_w.value),
                f0          = float(_lms_f0_w.value),
                mu          = float(_lms_mu_w.value),
                n_harmonics = int(_lms_harm_w.value),
                block_size  = int(_lms_block_w.value),
            )

    _lms_run_btn._click_handlers.callbacks[:] = []
    _lms_run_btn.on_click(_lms_on_run)

    display(widgets.VBox([
        widgets.HTML(
            "<b>LMS Adaptive Powerline Canceller</b><br>"
            "Select recording and parameters, then click Analyze &amp; Plot.<br>"
            "All channels (Line L, Line R, Mic) are processed automatically."
        ),
        _lms_folder_w,
        widgets.HTML("<hr style='margin:6px 0'>"),
        widgets.HTML("<b>LMS parameters</b>"),
        _lms_f0_w,
        _lms_mu_w,
        _lms_harm_w,
        _lms_block_w,
        widgets.HTML("<hr style='margin:6px 0'>"),
        _lms_fft_w,
        _lms_run_btn,
        _lms_out,
    ]))